# Void Clipex bot (free Colab runner)

1. Left side key icon (Secrets) -> Add new secret -> Name: `TELEGRAM_BOT_TOKEN` -> Value: bot token -> Notebook access ON.
2. Runtime -> **Run all**.
3. Wait 5-8 minutes, then message your bot on Telegram.
4. Keep the tab open. Free Colab allows roughly 12 hours; after that come back and Run all again.


In [ ]:
#@title 1) Install tools (3-5 min)
import os, re, subprocess, urllib.request

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

sh("apt-get -qq update && apt-get -qq install -y ffmpeg xz-utils >/dev/null 2>&1")

if "v24" not in sh("node -v").stdout:
    idx = urllib.request.urlopen("https://nodejs.org/dist/latest-v24.x/").read().decode()
    m = re.search(r'href="(node-v24\.[\d.]+-linux-x64\.tar\.xz)"', idx)
    sh("curl -fsSL https://nodejs.org/dist/latest-v24.x/" + m.group(1) + " -o /tmp/node.tar.xz")
    sh("tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1")

print("node:", sh("node -v").stdout.strip())
print(sh("npm i -g openclaw@latest --no-audit --no-fund 2>&1 | tail -2").stdout.strip())
print("openclaw:", sh("openclaw --version 2>&1 | head -1").stdout.strip())
print(sh("pip install -q yt-dlp faster-whisper 2>&1 | tail -2").stdout.strip())
print("ffmpeg:", sh("ffmpeg -version | head -1").stdout.strip()[:60])
print("yt-dlp:", sh("yt-dlp --version").stdout.strip())

In [ ]:
#@title 2) Bot files + free model router
import json, os, subprocess

BUNDLE = "https://base44.app/api/apps/6afd5d7f94aa00438e512b21/files/mp/public/6afd5d7f94aa00438e512b21/558b84d84_openclaw-vm-bundletar.gz"

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(("ok  " if r.returncode == 0 else "err ") + cmd[:70])
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-500:])
    return r

if not os.path.isdir("/tmp/freellmapi"):
    run("git clone --depth 1 https://github.com/tashfeenahmed/freellmapi /tmp/freellmapi")
run("cd /tmp/freellmapi/server && npm install --no-audit --no-fund >/dev/null 2>&1")
run("cd /tmp/freellmapi/server && npm run build >/dev/null 2>&1")

run("curl -fsSL " + BUNDLE + " -o /tmp/b.tgz")
os.makedirs("/tmp/bx", exist_ok=True)
subprocess.run("tar -xzf /tmp/b.tgz -C /tmp/bx", shell=True)
run("mkdir -p ~/.openclaw && cp -rf /tmp/bx/oc/. ~/.openclaw/")
run("mkdir -p /tmp/freellmapi/server/data && cp -f /tmp/bx/freellm/freeapi.db* /tmp/freellmapi/server/data/")

TOKEN = ""
try:
    from google.colab import userdata
    TOKEN = userdata.get("TELEGRAM_BOT_TOKEN") or ""
except Exception:
    TOKEN = ""
if not TOKEN:
    TOKEN = input("Bot token paste karein: ").strip()

p = os.path.expanduser("~/.openclaw/openclaw.json")
cfg = json.loads(open(p).read().replace("/root", os.path.expanduser("~")))
tg = cfg.setdefault("channels", {}).setdefault("telegram", {})
tg["botToken"] = TOKEN
tg["enabled"] = True
json.dump(cfg, open(p, "w"), indent=2)
print("token saved:", bool(TOKEN))

In [ ]:
#@title 3) Start router + bot (cell chalta rehna chahiye)
import subprocess, time

subprocess.Popen("cd /tmp/freellmapi/server && PORT=3001 HOST=127.0.0.1 node dist/index.js > /tmp/freellm.log 2>&1", shell=True)
time.sleep(12)
print("router:", subprocess.run("curl -s -o /dev/null -w '%{http_code}' localhost:3001/api/auth/status", shell=True, capture_output=True, text=True).stdout)
print("Bot start ho raha hai... Telegram par hi bhejein (1-2 min).")

while True:
    p = subprocess.Popen(["openclaw", "gateway", "run", "--force"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("[bot] gateway exited, restarting in 10s")
    time.sleep(10)